# RNAchime: training and decoy evaluation

This notebook runs the two existing command-line entry points and displays their output:

1. `RNAchime/src/train.py`: train the HGT model and save the best checkpoint for each fold.
2. `RNAchime/src/decoy_evaluation/evaluate.py`: evaluate the saved checkpoints using internal decoys, external decoys, or both.

The example inputs are provided under `RNAchime/data/demo`. Run the cells from top to bottom after reviewing the configuration. Commands run in separate Python processes using the notebook kernel's interpreter. 

Running the training cell starts the configured training job. For a short workflow check, set `NUM_EPOCHS = 100` and `KFOLD = 1` before running it. A short run checks execution, not model quality.


## 1. Create and select the Conda environment

The project provides `RNAchime/environment.yml`, which declares the `rnachime` environment and its dependencies. Complete the following setup in a terminal before running the notebook's Python cells.

### 1.1 Create the environment from the YAML file

Open a terminal with `RNAchime` as the working directory, then run:

```bash
conda env create -f environment.yml
conda activate rnachime
```

The environment name comes from `name: rnachime` in `environment.yml`. A machine-specific `prefix:` entry is not needed; Conda can use its default environment directory. If `rnachime` already exists and contains the required packages, skip the creation command and activate it directly.

Check the active environment and Python version:

```bash
conda env list
python --version
```

An asterisk in the environment list marks the active environment. The supplied YAML specifies Python 3.10.12.

### 1.2 Register the notebook kernel

The YAML includes `ipykernel`. With `rnachime` activated, register its Python interpreter as a notebook kernel:

```bash
python -m ipykernel install --user --name rnachime --display-name "rnachime"
```

Open `src/rnachime_demo.ipynb` in Jupyter or VS Code and select the **rnachime** kernel. Restart the kernel if you changed environments while the notebook was open. The next cell prints the actual Python executable and library versions so you can confirm the selection.

Use `RNAchime` as the project root. Start Jupyter in `RNAchime` or open this notebook with `RNAchime/src` as its working directory. The next cell locates the project root from either location. All data, model, log, and evaluation paths are defined under this root.


In [ ]:
import shlex
import subprocess
import sys
from datetime import datetime
from pathlib import Path

import dgl
import pandas as pd
import torch
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'src':
    PROJECT_ROOT = PROJECT_ROOT.parent
if PROJECT_ROOT.name != 'RNAchime' or not (PROJECT_ROOT / 'src/train.py').is_file():
    raise FileNotFoundError('Start this notebook with RNAchime or RNAchime/src as the working directory.')
PYTHON = sys.executable

print(f'Python executable: {PYTHON}')
print(f'Python version: {sys.version.split()[0]}')
print(f'RNAchime project root: {PROJECT_ROOT}')
print(f'PyTorch: {torch.__version__}; DGL: {dgl.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU 0: {torch.cuda.get_device_name(0)}')


## 2. Configure paths and experiment settings

This example uses the existing annotated graphs and decoy dictionary under `RNAchime/data/demo`.

Demo models are saved under `RNAchime/results/demo/unimol_demo`. 

To evaluate an existing demo run, set `RUN_TRAINING = False` and choose its timestamp with `EXISTING_RUN_TIME`. 

For another saved experiment, update `SAVE_DIR` and `EXPERIMENT_NAME` to match its folders.


In [ ]:
DEMO_DATA = PROJECT_ROOT / 'data/demo'
ANNOTATED_DATA = DEMO_DATA / 'annotated/pockets_nx_unimol_demo_annot'
DECOY_DICT_PATH = DEMO_DATA / 'decoy/demo_unimol_decoy_dict.p'
SAVE_DIR = PROJECT_ROOT / 'results/demo'
EXPERIMENT_NAME = 'unimol_demo'
EVAL_OUTPUT_ROOT = PROJECT_ROOT / 'outputs/demo_decoy_evaluation'

RUN_TRAINING = False
EXISTING_RUN_TIME = '20261101_151405'
NUM_EPOCHS = 100
KFOLD = 1
BATCH_SIZE = 32
WORKERS = 0  
SEED = 11
OUTPUT_DIM = 512
CRITERION = 'HuberLoss'
HEAD_SIZE = [32, 32, 32]
NUM_HEADS = [1, 1, 1]
ATTRIBUTOR_DIMS = [32, OUTPUT_DIM]
LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.00001
DROPOUT = 0.2
POOL = 'max'
DELTA = 1.0
POS_WEIGHT = 1.0
EARLY_STOP = 15
SPLIT_DIR = None  
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'

EVAL_MODE = 'both'
EVAL_BATCH_SIZE = 128
EVAL_WORKERS = 0

MODEL_ROOT = SAVE_DIR / EXPERIMENT_NAME
EXPECTED_FOLDS = KFOLD if KFOLD > 1 else 1
SESSION_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
LOG_DIR = PROJECT_ROOT / 'outputs/notebook_logs' / SESSION_ID
LOG_DIR.mkdir(parents=True, exist_ok=True)

print(f'Model root: {MODEL_ROOT}')
print(f'Notebook logs: {LOG_DIR}')
print(f'Device: {DEVICE}; folds: {EXPECTED_FOLDS}; evaluation: {EVAL_MODE}')

def run_command(command, log_path):
    command = [str(value) for value in command]
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print(f'Working directory: {PROJECT_ROOT}', flush=True)
    print(f'Command: {shlex.join(command)}', flush=True)
    print(f'Log: {log_path}', flush=True)
    with log_path.open('w', encoding='utf-8') as log:
        log.write(f'Command: {shlex.join(command)}\n')
        log.flush()
        with subprocess.Popen(command, cwd=PROJECT_ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, encoding='utf-8', errors='replace', bufsize=1) as process:
            try:
                for line in process.stdout:
                    print(line, end='', flush=True)
                    log.write(line)
                    log.flush()
                return_code = process.wait()
            except BaseException:
                process.terminate()
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
                raise
    if return_code:
        raise RuntimeError(f'Command exited with code {return_code}. Read {log_path}')
    return log_path


## 3. Train the model

Run the next cell to train the model with the settings above. It calls `python -u -m src.train` with `RNAchime` as the working directory, displays live output, and saves the same output to `LOG_DIR / 'train.log'`. It waits for all configured folds to finish. Each fold saves the checkpoint with the lowest validation loss.

Set `RUN_TRAINING = False` to reuse existing models instead. The next section selects the training timestamp for evaluation. Interrupting the training cell terminates the child process.

In [ ]:
RUN_TRAINING = True
train_command = [
    PYTHON, '-u', '-m', 'src.train',
    '--annotated_data', ANNOTATED_DATA,
    '--save_dir', SAVE_DIR, '--name', EXPERIMENT_NAME,
    '--num_epochs', NUM_EPOCHS, '--kfold', KFOLD,
    '--batch_size', BATCH_SIZE, '--workers', WORKERS,
    '--seed', SEED, '--criterion', CRITERION, '--device', DEVICE,
    '--head_size', *HEAD_SIZE, '--num_heads', *NUM_HEADS,
    '--attributor_dims', *ATTRIBUTOR_DIMS,
    '--learning_rate', LEARNING_RATE, '--weight_decay', WEIGHT_DECAY,
    '--dropout', DROPOUT, '--pool', POOL,
    '--delta', DELTA, '--pos_weight', POS_WEIGHT, '--early_stop', EARLY_STOP,
]
if SPLIT_DIR is not None:
    train_command += ['--split_dir', Path(SPLIT_DIR).expanduser().resolve()]
train_command = [str(value) for value in train_command]

if RUN_TRAINING:
    previous_run_dirs = {path.resolve() for path in MODEL_ROOT.iterdir() if path.is_dir()} if MODEL_ROOT.is_dir() else set()
    TRAIN_LOG = run_command(train_command, LOG_DIR / 'train.log')
    new_run_dirs = {path.resolve() for path in MODEL_ROOT.iterdir() if path.is_dir()} - previous_run_dirs
    print(f'Training finished. Created {len(new_run_dirs)} fold directories.')
else:
    new_run_dirs = set()
    print(f'Training skipped. Existing model root: {MODEL_ROOT}')


## 4. Select one complete training run

For newly trained models, the next cell selects only folders created by the training cell above. For existing models, it uses `EXISTING_RUN_TIME`, or infers the timestamp only when exactly one complete run is available. 

Fold numbering starts at zero. A five-fold run must have checkpoints, metadata, and split files for folds 0–4. For a single 80/20 run, the evaluator receives `--kfold 1` even if training used `KFOLD = 0`.


In [ ]:
if not MODEL_ROOT.is_dir():
    raise FileNotFoundError(MODEL_ROOT)

def fold_directories(timestamp):
    return [MODEL_ROOT / f'{EXPERIMENT_NAME}_{fold}_{timestamp}' for fold in range(EXPECTED_FOLDS)]

def complete_run(timestamp):
    folders = fold_directories(timestamp)
    return all((folder / 'meta.p').is_file() and (folder / f'splits_{fold}.p').is_file()
               and (folder / f'{folder.name}.pth').is_file() for fold, folder in enumerate(folders))

prefix = f'{EXPERIMENT_NAME}_0_'
search_dirs = new_run_dirs if RUN_TRAINING else {path for path in MODEL_ROOT.iterdir() if path.is_dir()}
available_times = sorted(path.name[len(prefix):] for path in search_dirs
                         if path.name.startswith(prefix) and complete_run(path.name[len(prefix):]))
print(f'Complete timestamps: {available_times}')

if not RUN_TRAINING and EXISTING_RUN_TIME is not None:
    RUN_TIME = str(EXISTING_RUN_TIME)
    if RUN_TIME not in available_times:
        raise ValueError(f'No complete run found for EXISTING_RUN_TIME={RUN_TIME}. Check MODEL_ROOT and KFOLD.')
elif len(available_times) == 1:
    RUN_TIME = available_times[0]
else:
    raise ValueError('Expected one complete run. Set EXISTING_RUN_TIME when reusing multiple saved runs, or check training output.')

RUN_DIRS = fold_directories(RUN_TIME)
EVAL_OUT_PATH = EVAL_OUTPUT_ROOT / EXPERIMENT_NAME / RUN_TIME
print(f'Selected timestamp: {RUN_TIME}')
print(f'Evaluation output directory: {EVAL_OUT_PATH}')
for fold, folder in enumerate(RUN_DIRS):
    print(f'Fold {fold}: {folder}')


## 5. Evaluation 

The evaluator loads each fold's best checkpoint and predicts only its saved test graphs. Both evaluation modes share one prediction pass per fold.

| `EVAL_MODE` | Candidate pool |
| --- | --- |
| `internal` | Other ligand vectors collected from the annotated graph directory.  |
| `external` | Decoys from `DECOY_DICT_PATH`; this path is required. |
| `both` | Evaluate both pools and write one result file for each. |

The next cell starts the real evaluation and waits for it to finish. Output includes the selected model folder, evaluated/skipped counts for each fold, and the mean rank, rank standard deviation, and output filename for each mode.

In [ ]:
evaluation_command = [
    PYTHON, '-u', '-m', 'src.decoy_evaluation.evaluate',
    '--annot_graph', ANNOTATED_DATA, '--model_root', MODEL_ROOT,
    '--out_path', EVAL_OUT_PATH, '--mode', EVAL_MODE,
    '--time', RUN_TIME, '--kfold', EXPECTED_FOLDS,
    '--device', DEVICE, '--batch_size', EVAL_BATCH_SIZE, '--workers', EVAL_WORKERS,
]
if EVAL_MODE in {'external', 'both'}:
    evaluation_command += ['--decoy_dict_path', DECOY_DICT_PATH]
evaluation_command = [str(value) for value in evaluation_command]

EVAL_LOG = run_command(evaluation_command, LOG_DIR / f'evaluate_{EVAL_MODE}.log')
print(f'Evaluation finished. Log: {EVAL_LOG}')


## 6. Report per-fold and final mean ranks

For each selected evaluation mode, compute the mean rank within each fold. The last row reports the arithmetic mean of the fold means, giving each fold equal weight. Higher rank scores indicate better ranking performance. 


In [ ]:
selected_modes = ['internal', 'external'] if EVAL_MODE == 'both' else [EVAL_MODE]
evaluation_summary = {}
for mode in selected_modes:
    result_path = EVAL_OUT_PATH / f'{MODEL_ROOT.name}_{RUN_TIME}_{mode}.txt'
    results = pd.read_csv(result_path)
    evaluation_summary[mode] = results.groupby('fold')['rank'].mean()

evaluation_summary = pd.DataFrame(evaluation_summary).rename_axis('fold')
evaluation_summary.loc['final_mean'] = evaluation_summary.mean()
display(evaluation_summary)
